In [ ]:
import shutil
import matplotlib.pyplot as plt
from transformers import set_seed
import torch

from attacks.bias_field.attack import attack_bf
from attacks.bias_field.visualization import plot_bias_field_attack, plot_progressive_bias_field_attack
from common.io import find_project_root, load_config, load_samples
from common.model import (
    extract_answer_token_ids,
    load_model,
    run_model,
    extract_answer
)
from common.preprocess import load_image_tensor, tensor_to_pil

In [ ]:
EXPERIMENT_NAME = "cps_8_eps_0p5"
SAMPLE_INDEX = 31
EVAL_STEPS_OVERRIDE = None
DEVICE = "cuda"
OVERWRITE = False
VERBOSE = 1

## Load Sample

In [ ]:
project_root = find_project_root()

sample_root = project_root / "data" / "OmniMedVQA" / "sample_mri"
question_path = sample_root / "question.json"
output_dir = project_root / "result" / "MedVLM-R1" / "debug_bias_field_attack"

# Overwrite + create output directories
if OVERWRITE:
    shutil.rmtree(output_dir, ignore_errors=True)
output_dir.mkdir(parents=True, exist_ok=True)
debug_dir = {
    "bias_field_directory": output_dir / "bias_field",
    "attacked_image_directory": output_dir / "attacked_image",
    "difference_directory": output_dir / "difference",
}
for directory in debug_dir.values():
    directory.mkdir(parents=True, exist_ok=True)

config_path = project_root / "configs" / f"{EXPERIMENT_NAME}.yaml"
shutil.copy2(config_path, output_dir / "config.yaml")

config = load_config(config_path)
experiment_config = config["experiment"]
model_config = config["model"]
attack_config = dict(config["attack"])
bias_config = config["bias_field"]

if EVAL_STEPS_OVERRIDE:
    attack_config["eval_step"] = int(EVAL_STEPS_OVERRIDE)

sample = load_samples(
    question_path,
    modality="MRI",
    start_index=SAMPLE_INDEX,
    end_index=SAMPLE_INDEX + 1,
    overwrite=True)[0]


In [ ]:
sample

## Load model

In [ ]:
set_seed(experiment_config.get("seed", 42), deterministic=True)

model, processor, generation_config = load_model(model_config)

answer_token_ids = extract_answer_token_ids(processor.tokenizer, verbose=VERBOSE)

## Clean inference

In [ ]:
question_id = str(sample["id"])
image_path = sample_root / sample["image"][0]

image = load_image_tensor(image_path)

problem = sample["problem"]
solution = sample["solution"]

print(f"Question ID: {question_id}")
print(f"Question: {problem}")
print(f"Solution: {solution}")

plt.figure(figsize=(4, 4))
plt.imshow(image.detach().cpu()[0], cmap="gray", vmin=0, vmax=1)
plt.title("Original Image")
plt.axis("off")
plt.show()

clean_output = run_model(
    question=problem,
    image=image_path,
    model=model,
    processor=processor,
    generation_config=generation_config,
)

print("Clean output:")
print(clean_output)

## Attack

In [ ]:
best_candidate, attack_history = attack_bf(
    image=image,
    problem=problem,
    target=solution,
    reference_output=clean_output,
    model=model,
    processor=processor,
    generation_config=generation_config,
    answer_token_ids=answer_token_ids,
    attack_config=attack_config,
    bias_config=bias_config,
    verbose=VERBOSE,
    debug=True,
    debug_dir=debug_dir,
)

In [ ]:
plot_bias_field_attack(
    image=image.detach().cpu().float(),
    bias_field=best_candidate["bias_field"].detach().cpu().float(),
    biased_image=best_candidate["image"].detach().cpu().float(),
    epsilon=bias_config["epsilon"],
    step=best_candidate["step"],
    loss=best_candidate["loss"],
    predicted_answer=best_candidate["answer"],
    attack_success=best_candidate["attack_success"],
    debug_dir=debug_dir,
)

print(f"Solution: {solution}")
print(f"Adversarial answer: {best_candidate['answer']}")
print(f"Attack success: {best_candidate['attack_success']}")
print(f"Best step: {best_candidate['step']}")
print(f"Best loss: {best_candidate['loss']:.6f}")
print(best_candidate["image_loss"])


## Validate attack

In [ ]:
# image = load_image_tensor(image_path)
bias_field_path = debug_dir["bias_field_directory"] / f"bias_field_{best_candidate['step']}.pt"
bias_field = torch.load(bias_field_path, map_location=image.device,)
reconstructed_image = torch.clamp(image * bias_field, min=0, max=1)

fig, axes = plt.subplots(1, 2, figsize=(6, 3))

axes[0].imshow(tensor_to_pil(image))
axes[0].set_title("Clean Image")
axes[0].axis("off")

axes[1].imshow(tensor_to_pil(reconstructed_image))
axes[1].set_title(f"Reconstructed Image - Step {best_candidate['step']}")
axes[1].axis("off")

plt.tight_layout()
plt.show()

adversarial_output = run_model(
    question=problem,
    image=reconstructed_image,
    model=model,
    processor=processor,
    generation_config=generation_config,
)

adversarial_answer = extract_answer(adversarial_output, tag="answer")

print("Adversarial output")
print(adversarial_output)
print(f"Adversarial answer: \033[31m{adversarial_answer}\033[0m")
print(f"Solution: \033[32m{solution}\033[0m")

In [ ]:
columns = 7

plot_progressive_bias_field_attack(
    image=image.detach().cpu().float(),
    bias_field_directory=debug_dir["bias_field_directory"],
    epsilon=bias_config["epsilon"],
    columns=columns
)
